# Lab 2: Decision Trees & Ensembles

**Due: February 10th at 11:59pm**

## Learning Goals
This lab is intended to introduce reading data from files, the use of discrete data, and the implementation of decision trees, both individually and in ensembles.  As with any programming assignment, you'll also be practicing and improving your general CS skills, like problem decomposition, algorithmic thinking, implementation and testing, language syntax, etc..  Here are some of the specific things you should practice while completing this assignment:

* How to load and work with data from a CSV file using the Pandas library
* How to implement the ID3 Decision Tree induction algorithm
* How to implement an ensemble of decision trees using the Bagging algorithm
* How to test algorithms using repeated hold-out validation

The assignment is presented in the form of an interactive Python notebook; it contains a mix of pre-made examples to help you understand how to do things, scaffolding with missing parts where you'll write your own code, and written short-answer questions.  Your job is to fill in answers to the written questions (which may require you to modify scaffolding code and re-run it, or may require you to write your own code and run it), as well as write the indicated functions to complete the programming portion of the assignment.  Look for the red <span style="color:red">TODO: ...</span> markers to help you spot places you'll need to complete things.

Please be sure to ***remove* the TODO markers** as you complete each aspect (i.e. don't leave a TODO that's for something you've actually done, that's poor style because it's confusing to anyone reading your code/documentation/etc.)

This lab is intended to be done with a partner (i.e. teams of two); partners will be assigned based on the survey

You'll have approximately **two weeks** to complete this lab, so be sure to start early and plan properly to get it all done in a timely fashion.  It's recommended that you start at the top and work your way down (i.e. the later parts are more difficult than the earlier ones).  As with any longer term assignment, it's a chance to train your time management skills.

# Constructing Decision Trees

Nearest Neighbor is a fine starting point, but no single algorithm is going to work well for all types of problems.  As a result, we'll need to use different algorithms for different problems.

As a case in point, Nearest Neighbor typically doesn't work very well for problems where the features are *discrete* variables (as opposed to *continuous*).  If the features are *ordinal*, they're discrete but at least ordered (e.g. "small", "medium", "large"), in which case we might be able to come up with some sort of meaningful definition of a `distance()` function.  However, for *cataogrical* features there's no ordering (e.g. "red", "blue", "green"), so really all we can say is that the feature is the same or different.

Fortunately, Decision Trees (DTs) handle discrete features very naturally; in fact, it requires a bit of extra work to get DTs to handle continuous data (it's not all *that* hard, but you won't be asked to do it for this assignment).

Note that SciKit Learn uses Numpy for all its data representations, which means it's actually not very good at handling discrete features.  As a result, while SKLearn does include a Decision Tree classifier, we can't actually make use of it as a baseline here, since we would first need to convert our data into numeric vectors to be able to have it run (and at that point, we wouldn't get the same result).  I've put the scores my reference implementation got down near the testing code so you've got a baseline anyway.

We'll also be using this as an excuse to learn about reading data from files (instead of just using data sets that come with SKLearn).

***


## Reading data

The first thing we need to do is read some data.  Previously, we just used a data set that was built in to SciKit Learn, but this time we'll be reading a data file from disk.

This week, we'll use a library called **Pandas** to read our data; it's really good for reading things like excel spreadsheets and comma-separated-value (CSV) data files.  

Like SciKit Learn, Pandas also has quite good documentation: https://pandas.pydata.org/docs/

Also, **as a reminder, the warmup lab contains a number of explanations and examples of Pandas usage**, so you may want to use code snippets from there as a starting point.

## The Data
For this part of the assignment, you'll be using a dataset covering congressional voting behavior, one looking at re-occurence of breast cancer, and one on classifying edibility of mushrooms.  The original versions are downloadable from the UCI machine learning repository: 
* https://archive.ics.uci.edu/ml/datasets/Congressional+Voting+Records 
* http://archive.ics.uci.edu/ml/datasets/Mushroom
* https://archive.ics.uci.edu/ml/datasets/Breast+Cancer

For the raw data, you can just use the copy that's included with the assignment scaffolding.  However, to get the full description of the dataset (which you'll need for the written questions at the bottom), you'll need to click on those links, and then read about the  data set.
***

### Using Pandas to load data

We'll use the `read_csv()` method to load the files; `header=none` means that the first line of the file is a normal example (as opposed to a special header with column names), and we'll use a list of strings to specify the attribute names.  Then, we just need to give it the path to the datafile and it will be loaded as a Pandas dataframe.

Pandas dataframes work a lot like relational databases; there's a ton of things you can do with Pandas, but for today we'll keep things relatively simple.

You can read more about the `read_csv()` function here: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

**HINT:** _some of the examples of Pandas usage in the warmup lab may be helpful for doing this lab!_

#### Congress Data Set

In [1]:
# this time we'll use Pandas for reading our data file (it's great for CSVs)
import pandas as pd

# the data file doesn't have a headder, so we'll build an array of strings holding the column names
congressFeatureNames = ['party']  # the first column is political party
for i in range(16):
    congressFeatureNames.append('vote-'+str(i))  # remaining columns are votes for different bills

# this next line actually reads the file in as a Pandas dataframe
congressData = pd.read_csv('house-votes-84.data', header=None, names=congressFeatureNames )

In [2]:
# as usual, len will give us the number of elements in a collection (in this case, how many examples are in our dataset)
len(congressData)

# congressData

435

#### Mushroom Data Set

In [3]:
mushroomFeatureNames = ['edible', 'cap-shape', 'cap-surface', 'cap-color', 'bruises', 'odor', 
                'gill-attachment', 'gill-spacing', 'gill-size', 'gill-color', 'stalk-shape', 'stalk-root', 
                'stalk-surface-above-ring', 'stalk-surface-below-ring', 'stalk-color-above-ring', 'stalk-color-below-ring', 
                'veil-type', 'veil-color', 'ring-number', 'ring-type', 'spore-print-color', 'population', 'habitat']
    
mushroomData = pd.read_csv('agaricus-lepiota.data', header=None, names=mushroomFeatureNames)
len(mushroomData)
# mushroomData

8124

#### Breast Cancer Data Set

In [4]:
breastcancerFeatureNames = ['class', 'age', 'menopause', 'tumor-size', 'inv-nodes', 'node-caps', 
                            'deg-malig', 'breast', 'breast-quad', 'irradiated']
breastcancerData = pd.read_csv('breast-cancer.data', header=None, names=breastcancerFeatureNames)
len(breastcancerData)
# breastcancerData

286

# Part 1: Questions about the data

Modify this markdown cell to add the answers to the following short-answer questions.  In doing so, make use of the code you've written above (as well as the provided code).  _**Note that you won't be able to answer the final few questions until after finishing your implementation.**_

1. What is the _full name_ (i.e. human-readable, not filename) of the 'congress' data set?

 "Congressional Voting Records" is the official name.

2. What sort of real-world user might be interested in a system that could successfully solve this classification problem?

It could be really useful for a political researcher who is trying to predict the number of states that are democratic and that are republican for the upcoming elections based on the responses of random sample of people from each state.  

Additionally, campaigners model could use this data to identify people's political affiliation and employ targeted advertising to improve the chances of winning an election.

3. What are the stakes for this problem?  In other words, who might be hurt if the system makes a mistake?  How bad are the consequences?

If the model was inaccurate, the political party predictions may be wrong. This could lead to distrust against political researcher because the people who trusted them with accurate data of how the political landscape of America is inaccurate. 

If campaigners use an inaccurate model, then campaigners could be seen as wasting resources for an election and ultimately lose funding and support.

The consequences are moderately high because of its polticial consequences.  

4. How many features (not including the class label) does each example in the data set have?

For Congressional Voting Records dataset, there are 16 features.

5. How many examples does the data set contain?

435 examples

6. What are the available class labels? Give both the encoding in the data set (i.e. the raw value) and the human-readable label associated with each value.

The raw values are: republican, democrat
These values should be human readable.

---

7. What is the _full name_ (i.e. human-readable, not filename) of the 'mushroom' data set?

Mushroom

8. What sort of real-world user might be interested in a system that could successfully solve this classification problem?

People who collect mushrooms might want this dataset to predict how poisonous the mushrooms they collected are before selling it to the market.

9. What are the stakes for this problem?  In other words, who might be hurt if the system makes a mistake?  How bad are the consequences?

If this model is inaccurate, then these mushrooms could kill people. If there are false positives in the model, it could lead to a loss in revenue.

10. How many features (not including the class label) does each example in the data set have?

22 features

11. How many examples does the data set contain?

8124

12. What are the available class labels? Give both the encoding in the data set (i.e. the raw value) and the human-readable label associated with each value.
The raw class labels are: p,e
The human readable for each of them are poisonous and edible respectively.
---

13. What is the _full name_ (i.e. human-readable, not filename) of the 'breast-cancer' data set?

Breast Cancer

14. What sort of real-world user might be interested in a system that could successfully solve this classification problem?

Doctors might be interested in this to identify the risk of reoccurrence of breast cancer.

15. What are the stakes for this problem?  In other words, who might be hurt if the system makes a mistake?  How bad are the consequences?

If this system is incorrect, then patients might get the wrong idea of how they should react against possible breast cancer reoccurrence.

16. How many features (not including the class label) does each example in the data set have?

9 features

17. How many examples does the data set contain?

286

18. What are the available class labels? Give both the encoding in the data set (i.e. the raw value) and the human-readable label associated with each value.

The raw labels are: no-recurrence-events, recurrence-events
These labels should be human readable.

---

19. For each dataset, describe in a sentence or two what the final tree looks like.  For example, how would you describe the "shape" of the tree?  Is it narrow and deep? Broad and shallow?  etc.

For the Congressional Voting Records dataset, the tree is probably shallow and narrow because politics is very polar so it only takes a few feature to have a high confidence of predicting correctly. It is narrow because each feature is boolean

For Mushroom dataset, the tree is probably deep and broad because it is hard to tell which mushrooms are posionious based on a few features and each feature multiple category.

For the Breast Cancer dataset, the tree could also be deep and broad because it is detecting the relationships between each feature is complex and each feature has multiple category.

20. For each dataset, what does this tell you about the problem?

The dataset for Congressional Voting might be simpler and easier. The Mushroom and Breast Cancer dataset could be more complex and have a higher risk of overfitting.

21. Which of the datasets do decision trees perform better on?

Congressional Voting dataset might be very accurate because each feature have a clear relationship with each political party.
The Mushroom and Breast Cancer dataset is more difficult so the decision trees might be not as accurate.

22. Why?  What about that data makes it more well suited to decision tree modeling?

Decision trees with a clear relationship with target values are the most well suited for decision trees and performs the best.

# Part 2: implementing the ID3 Decision Tree Learning algorithm

## Scaffolding

Here we'll provide some scaffolding to get you started, including some recommended function headers.  The final implementation of the algorithm is up to you; you can modify the scaffolding as needed, just try not to make the problem more complicated than it needs to be.

It's _strongly recommended_ (but not required) that you leave the data in the Pandas dataframe, and work with it using Pandas operators, since this will automatically make use of views (i.e. you get references rather than making deep copies of the example sets).  Whatever you use, try as always to make your code reasonably efficient (in big-O terms).

You should come up with your own development plan here; it should probably look very similar to the one for implementing the KNN algorithm, but with different functions.

## Defining Trees
Here's a basic TreeNode class; it's pretty minimal, but you don't really need anything more for this algorithm.
- `isLeaf` is meant to be a boolean, set to True if the node is a leaf node, and False otherwise
- `val` is normally the attribute (i.e. column identifier) that this node splits on, but for a leaf node it's used as the output class label instead (note that here we use two separate member variables for these two things, but since any given node will only ever use one of them, you _could_ just have the one member variable; it would save a little space, but come at the cost of clarity)
- `children` is a dictionary, which maps from attribute values to tree nodes.  For non-leaf nodes, it should have an entry for each value that the attribute this nodes splits on can take.  Since those attribute values are used directly as the keys for the dictionary, tree traversal is very simple.  For a leaf node, it should remain empty.

Remember that members are all public in Python, so we don't need any getter/setter methods; in fact, the constructor is the only method we'll bother defining for this class.

In [5]:
class DecisionTreeNode:
    def __init__(self, leaf, val):
        self.isLeaf = leaf
        if (self.isLeaf):
            self.label = val
        else:
            self.splitAttribute=val
        self.children = {}

# first we define a tree-node type
# class DecisionTreeNode:
    # constructor sets up members
  #  def __init__(self, leaf, val):
        # initialize instance members with specified values
    #    self.isLeaf = leaf  # is this node a leaf node?
   #     if (self.isLeaf):
     #       self.label = val  # if it's a leaf, what's the class label associated with it; 
      #  else:
       #     self.splitAttribute = val   # if it's a non-leaf, what attribute do we split on
            
        #self.children = {}  # list of child nodes (will be filled later if this node is not a leaf)

### Examining trees
Assuming you use the above node structure as intended, the following method should give a nice display of a decision tree; just call `printDecisionTree(root)` on the root node of the tree

In [6]:
# a recursive function to pretty-print a decision tree; 
# tries to match the visual style of decision-tree-printing in SciKit Learn
def printDecisionTree(node, indent =0, val =0):
    if (node == None): # this case should never fire for a properly built decision tree
        print('!!empty node!!')
        return
    if (node.isLeaf): # if it's a leaf node, we just need to print the class label
        for i in range(indent): # make sure we get the indent level right
            print("|   ", end="")
        print("|--- class: ", node.label)
    else: # for non-leaf nodes, we need to loop through our children and print each of them
        for val, child in node.children.items():
            for i in range(indent): # indent properly
                print("|   ", end="")
            print("|--- ", node.splitAttribute, " = ", val) # print which attribute/value this child is associated with
            printDecisionTree(child, indent+1, val) # recursively print the sub-tree for that child node


### Evaluating Trees
As usual, we don't want to use the same data for training and testing, so here's a basic function to shuffle and split our data into train and test sets, build a tree using the training data, evaluate it on the testing data, and then show the tree itself using the previous function.

**NOTE** that this function won't do anything terribly interesting until you fill in the `buildDecisionTree()` and `testDecisionTree()` functions, and you need to use the expected interface for them if you want this function to work correctly.

This is a helper function that you're encouraged to use to help test your code; example calls are given below the relevant function definition stubs.  You **shouldn't need to modify** this code unless you change the class definition given above, though it's fine to modify it if you do need to.

In [7]:
# a helper function to training & test decision tree algorithms
# params:
#   data - full dataset (will be split into test & train)
#   trainFrac - fraction of data to use for training (remainder will be used for testing)
#   seed - seed value for random number generator; defaults to 0 for repeatability, but try other values!
#   printTree - whether or not to show the tree (if False function just returns the accuracy)
def trainAndTestDT(data, trainFrac =0.7, seed =0, printTree =True):
    # let's shuffle this and split it into train and test sets:
    shuffled = data.sample(frac=1, random_state=seed) # randomly re-order the examples
        # note: the 'frac=1' means use all the examples; also note this creates a new "view", 
        # it doesn't do a deep copy, so the row index values in `shuffled` will be out of order
    trainCount = int(len(data) * trainFrac) # figure out how many examples we want in each set
    train = shuffled[:trainCount] # slice out the examples we'll use for training (again, creates a view)
    test = shuffled[trainCount:] # use the remainder for testing
    
    root = buildDecisionTree(train, train.columns[0]) # call the recursive tree building algorithm on the full data set
    acc = testDecisionTree(root, test) # test the tree it gave us
    
    if (printTree):
        printDecisionTree(root) # display the tree that we got
        print("Test set accuracy: ", acc) # print accuracy

    return acc

### Repeated testing
To try to get a better sense of how reliable or consistent your algorithm is, you may want to try running it multiple times, using different splits of the data each time.  By passing in different values for the RNG seed, this function will run the above `trainAndTestDT()` function on different train/test splits and print the output each time.

This method of testing is referred to as ***repeated hold-out validation***, since we're trying to *validate* our results by *repeatedly* applying the *hold-out* testing method (i.e. randomly splitting our data into fixed-size 'train' and 'test' portions).

Again, you shouldn't need to modify this function; you'll use it later on once you've written your parts of the implementation.

In [8]:
# helper function to run the train & test function multiple times
# params:
#  data - full data set
#  repeats - number of times to repeat the experiment with a different RNG seed
def testRepeated(data, repeats):
        total=0 # accumulator variable for the accuracies so we can calculate the mean
        for j in range(repeats):
            # train and test using j as the seed for the RNG
            acc = trainAndTestDT(data, 0.7, j, False)
            print("\t Acc: %.4f" % (acc))
            total += acc
        print("Mean accuracy: %.4f" % (total / repeats)) 

***
## Recursive Tree Building from scratch
Here you will write a recursive tree-building algorithm using Information Gain to decide how to split the data at each node.  Function headers have been provided to indicate a suggested way of decomposing this problem, but it's ultimately up to you how you solve it.  **Be sure you know what a function is supposed to do before you write it**, particularly for functions that are intended as 'helper' functions.

_Note:_ if you use the Pandas dataframes directly, you need to be careful about iteration, because the random shuffle/split done to get training and testing data will result in sets that don't have row indexes the way you expect (the row index values are still based on the original data, not the _view_ you're using as the training set).  Thus, you may want to use something like `iterrows` (see: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.iterrows.html) so you don't get indexing errors.

However, it is usually much easier and more efficient to rely on Pandas to do the heavy lifting, rather than trying to use `iterrows` to go through the table by hand.  See the examples of using Pandas for counting and calculating ratios that were provided in the warmup lab for ideas about what might be possible.

**It is _strongly recommended_ you come up with a development plan, including a top-down design and a plan for testing your functions, _before_ you start actually writing code.**


In [9]:
#       Should take in a set of examples and an identifier for which column 
#         (aka 'attribute', aka 'feature') to use for splitting.
#       Should return several subsets of data, one for each value of that attribute.
#       Note that you can decide what type to return based on how the rest of your program
#       works; I used a dictionary that maps attribute values to dataframes.

# note: If using Pandas, the easiest way is to specify the attribute using the column
#       name, since you can use that as an 'index' value to grab the desired column of values;
#       see the warmup lab Pandas section for examples
def split(examples, columnID):
    splitData = {}
    featureValues = examples[columnID].unique()
    for featureValue in featureValues:
        splitData[featureValue] = examples[examples[columnID] == featureValue]
    return splitData

In [10]:
def helper_print_split_data(splitData):
    for key, item in splitData.items():
        print(f"key={key}\nitem: {item}\n\n")

In [11]:

# You'll want to write several tests; you're encouraged to create small examples to use for
# testing.  As an example, two extra data files are provided, which we load here; feel free
# to use these for testing purposes.  Again, this is an *example* of how to do testing, not
# an indication that two test cases is sufficient.

# Keep in mind that for the following code to work correctly, you may need to modify things for 
# your codebase (e.g. if you don't use raw pandas tables as input, this won't work right)
uniformData = pd.read_csv('uniformTest.data', header=None, names=['label', 'feature1'])
evenSplitData = pd.read_csv('evenSplitTest.data', header=None, names=['label', 'feature1'])

# Note that you can modify things to play with different values.  You can either
# edit the CSV file, or just modify the table in memory, e.g.:
evenSplitData.insert(2,"feature2",['y','y','y','y'])
evenSplitData.insert(3,"feature3",['y','y','y','n'])
evenSplitData.insert(4,"feature4",['n','y','y','y'])

print(uniformData)
print("-----------------------------------------------------")
helper_print_split_data(split(uniformData, 'feature1'))
print("-----------------------------------------------------")
print(evenSplitData)
print("-----------------------------------------------------")
helper_print_split_data(split(evenSplitData, 'feature2'))
print("-----------------------------------------------------")
helper_print_split_data(split(evenSplitData, 'feature4'))

  label feature1
0   red        y
1   red        y
2   red        y
3   red        y
-----------------------------------------------------
key=y
item:   label feature1
0   red        y
1   red        y
2   red        y
3   red        y


-----------------------------------------------------
  label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
2  blue        n        y        y        y
3  blue        n        y        n        y
-----------------------------------------------------
key=y
item:   label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
2  blue        n        y        y        y
3  blue        n        y        n        y


-----------------------------------------------------
key=n
item:   label feature1 feature2 feature3 feature4
0   red        y        y        y        n


key=y
item:   label feature1 feature2 featur

In [12]:
print(evenSplitData)

print("\n Feature 1 splits y and n")
print(split(evenSplitData, 'feature1')['y'])
print(split(evenSplitData, 'feature1')['n'])

print("\n Feature 2 splits y and n")
print(split(evenSplitData, 'feature2')['y'])
try:
    print(split(evenSplitData, 'feature2')['n'])
except KeyError as e:
    print(e)

print("\n Feature 3 splits y and n")
print(split(evenSplitData, 'feature3')['y'])
print(split(evenSplitData, 'feature3')['n'])

print("\n Feature 4 splits y and n")
print(split(evenSplitData, 'feature4')['y'])
print(split(evenSplitData, 'feature4')['n'])

  label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
2  blue        n        y        y        y
3  blue        n        y        n        y

 Feature 1 splits y and n
  label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
  label feature1 feature2 feature3 feature4
2  blue        n        y        y        y
3  blue        n        y        n        y

 Feature 2 splits y and n
  label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
2  blue        n        y        y        y
3  blue        n        y        n        y
'n'

 Feature 3 splits y and n
  label feature1 feature2 feature3 feature4
0   red        y        y        y        n
1   red        y        y        y        y
2  blue        n        y        y        y
  label feature1 feature2 feature3 

In [13]:
from math import log
#       this should make use of your `split()` function
#       Note that you'll likely want to pass in the target (or label) column
#       as the columnID here, but the math should work for any column.

def entropy(examples, columnID):
    entropy = 0
    splitData = split(examples, columnID)
    for className, dataFrames in splitData.items():
        p_i = len(dataFrames)/len(examples)
        entropy += p_i * log(p_i, 2)
    return -entropy if entropy else 0 # just because seeing "-0" is annoying!

In [14]:
evenSplitData

,label,feature1,feature2,feature3,feature4
0,red,y,y,y,n
1,red,y,y,y,y
2,blue,n,y,y,y
3,blue,n,y,n,y


In [15]:

# than this again, modify these as appropriate for your codebase
# Note that you should add similar testing cells for each of the functions below!
print(entropy(uniformData, 'feature1')) # should be 0
print(entropy(evenSplitData, 'feature1')) # should be 1

print(entropy(evenSplitData, 'feature2')) # should be 0
print(entropy(evenSplitData, 'feature3')) # should be 0.811
print(entropy(evenSplitData, 'feature4')) # should be 0.811

0
1.0
0
0.8112781244591328
0.8112781244591328


In [16]:
def informationGain(examples, testColumn, targetColumn):
    entropyParent = entropy(examples,targetColumn)
    splitLabel = split(examples,testColumn)
    entropyChild = 0
    for label, splitDataFrame in splitLabel.items():
        weight = len(splitDataFrame)/len(examples)
        entropyChild += weight*entropy(splitDataFrame,targetColumn)
    return entropyParent - entropyChild

In [17]:
for i in range(1,5):
    print(f"IG({i})=",informationGain(evenSplitData,'label','feature'+str(i)))

IG(1)= 1.0
IG(2)= 0.0
IG(3)= 0.31127812445913283
IG(4)= 0.31127812445913283


In [18]:
def plurality(examples, columnID):
    freq = {}
    max_element = ""
    max_count = 0
    for item in examples[columnID]:
        if(not item in freq):
            freq[item] = 1
        else:
            freq[item]+=1
        # after updating frequencies above, we calculate the maximum element immediately
        if(freq[item] > max_count):
            max_count = freq[item]
            max_element = item
    return max_element

In [19]:
def getNextInfoGain(examples, columnID):
    infoGain = {}
    for feature in (examples.columns):
        if feature is not columnID:
            infoGain[feature] = informationGain(examples,columnID,feature)
    
    return infoGain


In [20]:

def buildDecisionTree(examples, targetColumn):
    initial_IG = getNextInfoGain(examples, targetColumn)
    sorted_IG = dict(sorted(initial_IG.items(), key=lambda item: item[1], reverse=True))
    first_key = next(iter(sorted_IG))
    if(sorted_IG[first_key] == 0): # the maximum information gain is ZERO. It has to be a leaf
        plurality_label = plurality(examples, targetColumn)
        return DecisionTreeNode(True, plurality_label)    
    current_node = DecisionTreeNode(False, first_key)
    splitData = split(examples, first_key)
    for key, item in splitData.items():
        children_node = buildDecisionTree(item, targetColumn)
        current_node.children[key] = children_node 
    return current_node

In [21]:
buildDecisionTree(evenSplitData, "label")

In [22]:
import random  # if we have no information, we'll need to select randomly
def classify(node, example):
    if(node.isLeaf):
        return node.label
    current_class = example[node.splitAttribute]
    if(current_class in node.children):
        return classify(node.children[current_class], example)
    random_key = random.choice(list(node.children.keys()))
    return classify(node.children[random_key], example)

In [23]:

def testDecisionTree(root, testSet):
    label_name = testSet.columns[0]
    correct_answers = 0
    for index, row in testSet.iterrows():
        if(row[label_name] == classify(root, row)):
            correct_answers += 1
    return correct_answers / len(testSet)

***
#### Tests on Mushroom data:
- with default parameters, should be 100% accuracy
- should likely be 100% for each repeat
***

In [24]:
print("DT on Mushroom data:")
trainAndTestDT(mushroomData) 

DT on Mushroom data:
|---  odor  =  p
|   |--- class:  p
|---  odor  =  n
|   |---  spore-print-color  =  k
|   |   |--- class:  e
|   |---  spore-print-color  =  n
|   |   |--- class:  e
|   |---  spore-print-color  =  w
|   |   |---  habitat  =  w
|   |   |   |--- class:  e
|   |   |---  habitat  =  p
|   |   |   |--- class:  e
|   |   |---  habitat  =  g
|   |   |   |--- class:  e
|   |   |---  habitat  =  l
|   |   |   |---  cap-color  =  c
|   |   |   |   |--- class:  e
|   |   |   |---  cap-color  =  n
|   |   |   |   |--- class:  e
|   |   |   |---  cap-color  =  y
|   |   |   |   |--- class:  p
|   |   |   |---  cap-color  =  w
|   |   |   |   |--- class:  p
|   |   |---  habitat  =  d
|   |   |   |---  gill-size  =  n
|   |   |   |   |--- class:  p
|   |   |   |---  gill-size  =  b
|   |   |   |   |--- class:  e
|   |---  spore-print-color  =  o
|   |   |--- class:  e
|   |---  spore-print-color  =  b
|   |   |--- class:  e
|   |---  spore-print-color  =  y
|   |   |--- class:

1.0

In [25]:
testRepeated(mushroomData, 10) 

	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
	 Acc: 1.0000
Mean accuracy: 1.0000


***
#### Tests on Congress data:
 - with default parameters, should be 91.6% accuracy
 - mean of 10 repeats should be something like 92.7%
***

In [26]:
print("DT on Congress data:")
trainAndTestDT(congressData)

DT on Congress data:
|---  vote-3  =  n
|   |--- class:  democrat
|---  vote-3  =  y
|   |---  vote-10  =  n
|   |   |---  vote-14  =  y
|   |   |   |---  vote-9  =  y
|   |   |   |   |--- class:  republican
|   |   |   |---  vote-9  =  n
|   |   |   |   |---  vote-15  =  n
|   |   |   |   |   |--- class:  republican
|   |   |   |   |---  vote-15  =  y
|   |   |   |   |   |--- class:  democrat
|   |   |   |   |---  vote-15  =  ?
|   |   |   |   |   |---  vote-1  =  y
|   |   |   |   |   |   |--- class:  republican
|   |   |   |   |   |---  vote-1  =  n
|   |   |   |   |   |   |--- class:  democrat
|   |   |---  vote-14  =  n
|   |   |   |--- class:  republican
|   |   |---  vote-14  =  ?
|   |   |   |--- class:  republican
|   |---  vote-10  =  y
|   |   |---  vote-15  =  n
|   |   |   |---  vote-0  =  y
|   |   |   |   |--- class:  republican
|   |   |   |---  vote-0  =  n
|   |   |   |   |---  vote-9  =  y
|   |   |   |   |   |---  vote-2  =  y
|   |   |   |   |   |   |--- class:  de

0.916030534351145

In [27]:
testRepeated(congressData, 10) 

	 Acc: 0.9160
	 Acc: 0.9084
	 Acc: 0.9466
	 Acc: 0.9084
	 Acc: 0.9160
	 Acc: 0.9542
	 Acc: 0.9542
	 Acc: 0.9084
	 Acc: 0.9237
	 Acc: 0.9313
Mean accuracy: 0.9267


***
#### Tests on Breast Cancer data:
- mean of 10 repeats should be something like ~65%
***

In [28]:
print("DT on Breast Cancer data:")
trainAndTestDT(breastcancerData) 

DT on Breast Cancer data:
|---  inv-nodes  =  0-2
|   |---  tumor-size  =  40-44
|   |   |---  age  =  70-79
|   |   |   |--- class:  no-recurrence-events
|   |   |---  age  =  30-39
|   |   |   |---  deg-malig  =  1
|   |   |   |   |--- class:  recurrence-events
|   |   |   |---  deg-malig  =  2
|   |   |   |   |--- class:  no-recurrence-events
|   |   |---  age  =  50-59
|   |   |   |--- class:  no-recurrence-events
|   |   |---  age  =  60-69
|   |   |   |--- class:  recurrence-events
|   |   |---  age  =  40-49
|   |   |   |--- class:  no-recurrence-events
|   |---  tumor-size  =  35-39
|   |   |---  age  =  40-49
|   |   |   |---  breast-quad  =  left_low
|   |   |   |   |--- class:  no-recurrence-events
|   |   |   |---  breast-quad  =  left_up
|   |   |   |   |--- class:  recurrence-events
|   |   |---  age  =  20-29
|   |   |   |--- class:  no-recurrence-events
|   |   |---  age  =  30-39
|   |   |   |--- class:  recurrence-events
|   |   |---  age  =  50-59
|   |   |   |--- cl

0.6744186046511628

In [29]:
testRepeated(breastcancerData, 10) 

	 Acc: 0.6628
	 Acc: 0.6860
	 Acc: 0.6628
	 Acc: 0.6279
	 Acc: 0.7093
	 Acc: 0.6512
	 Acc: 0.6279
	 Acc: 0.6744
	 Acc: 0.6860
	 Acc: 0.5581
Mean accuracy: 0.6547


***
# Part 3: Ensembles of Trees

Decision trees are nice in many ways, but they have a tendency to overfit their training data.  One way to combat this tendency is to train multiple trees and then combine the results.

Here, you'll write code to create an ensemble of decision trees using **bagging**, short for **b**ootstrap **agg**regat**ing**.  This method works by creating several different training sets, training a decision tree on each set, and then letting the resulting trees "vote" on the correct answer to a novel query.

A function to train an ensemble should take a number of trees $k$ and the fraction of the training data $f$ to use for each tree.  It should then call your `buildDecisionTree()` function $k$ times, each time using $f$ percent of the full training set selected at random _with replacement_.  If you're using Pandas, you can look at the `sample()` method (https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.sample.html), which will do most of the work for you.

Note that you should not have to actually modify your `buildDecisionTree()` function; instead, you are just calling it repeatedly with different data sets, and storing those trees to make a 'forest'.

To classify a novel example using an ensemble, you'll need to call the single-tree classify function on that example for each tree in the ensemble; then, pick the label that got the most votes (i.e. the plurality choice) as the output for the ensemble as a whole.

Finally, you'll want to write a function to test your ensemble on the testing set and then try it out (you may want to copy and modify the `trainAndTestDT()` function given above).

Again, it is _**strongly recommended**_ you come up with a development plan, including a top-down design and a plan for testing your functions, _before_ you start actually writing code.

Note that you should typically see an improvement of a couple percentage points on average for the Congress and Breast Cancer data when using bagging (e.g. for Congress I got an average of 94.5% over 10 runs using 5 trees per ensemble and 0.5 as my bagging fraction, as opposed to the 92.7% for single DTs; for Breast Cancer I got 67.2% from 10 repeats of 10 trees with 0.7 bagging fraction, as opposed to 65.9% for single trees).  The mushroom data should already be getting 100% accuracy with a single tree, so there's really no room for improvement on that one.


In [30]:
def arrayPlurality(array):
    freq = {}
    max_element = ""
    max_count = 0
    for item in array:
        if not item in freq:
            freq[item] = 1
        else:
            freq[item] += 1
        # after updating frequencies above, we update our return as well
        if(freq[item] > max_count):
            max_count = freq[item]
            max_element = item
    return max_element

In [31]:
# k: nums trees
# f: percent of the full training set selected at random with replacement
def buildForest(examples, targetColumn, k, f):
    forest = []
    for i in range(k):
        random_sample = examples.sample(n=(int(len(examples)*f)))
        tree = buildDecisionTree(random_sample, targetColumn)
        forest.append(tree)
    return forest

buildForest(evenSplitData,'label',3, 0.75)

In [32]:
def testRandomForest(randomForest, testSet):
    label_name = testSet.columns[0]
    correct_answers = 0
    for index, row in testSet.iterrows():
        current_row_labels = []
        for tree in randomForest:
            current_row_labels.append(classify(tree, row))
        plurality_class = arrayPlurality(current_row_labels)
        if(plurality_class == row[label_name]):
            correct_answers += 1
    return correct_answers / len(testSet)
            
        

In [33]:
def trainAndTestForest(data, trainFrac = 0.7, seed = 0, printForest = True, treeNum = 10, baggingFrac=0.5):
    shuffled = data.sample(frac=1, random_state=seed)
    trainCount = int(len(data) * trainFrac)
    train = shuffled[:trainCount]
    test = shuffled[trainCount:]

    randomForest = buildForest(train, train.columns[0], treeNum, baggingFrac)
    acc = testRandomForest(randomForest, test)
    print("Test set accuracy: ", acc)
    if(printForest):
        for tree in randomForest:
            printDecisionTree(tree)
    return acc

In [34]:
trainAndTestForest(breastcancerData, printForest = False)


Test set accuracy:  0.6976744186046512


0.6976744186046512

In [35]:
def testRepeatedRandomForest(data, repeats, treeNum, baggingFrac):
    total = 0
    for j in range(repeats):
        acc = trainAndTestForest(data, trainFrac = 0.7, seed = j, printForest = False, treeNum = treeNum, baggingFrac = baggingFrac)
        total += acc
    print("Mean accuracy: %.4f" % (total / repeats))

In [36]:
testRepeatedRandomForest(breastcancerData, 10, 10, 0.7)

Test set accuracy:  0.6627906976744186
Test set accuracy:  0.6511627906976745
Test set accuracy:  0.7209302325581395
Test set accuracy:  0.6627906976744186
Test set accuracy:  0.6744186046511628
Test set accuracy:  0.7209302325581395
Test set accuracy:  0.6395348837209303
Test set accuracy:  0.6511627906976745
Test set accuracy:  0.6046511627906976
Test set accuracy:  0.6162790697674418
Mean accuracy: 0.6605


In [37]:
testRepeatedRandomForest(congressData, 10, 5, 0.5)

Test set accuracy:  0.9312977099236641
Test set accuracy:  0.9236641221374046
Test set accuracy:  0.9541984732824428
Test set accuracy:  0.9312977099236641
Test set accuracy:  0.9312977099236641
Test set accuracy:  0.9694656488549618
Test set accuracy:  0.9694656488549618
Test set accuracy:  0.9312977099236641
Test set accuracy:  0.9618320610687023
Test set accuracy:  0.9236641221374046
Mean accuracy: 0.9427


In [38]:
testRepeatedRandomForest(mushroomData, 10, 5, 0.5)

Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Test set accuracy:  1.0
Mean accuracy: 1.0000


***
## Optional extension

If you want to take things one step further, try implementing the full Random Forest algorithm, which lets you also choose a fraction of the _features_ to consider when creating a tree.  You can either do this by selecting features on a per-tree basis or a per-node basis; for my reference implementation, it was much easier to do it on a per-node basis (in fact, it only required a couple of extra lines of code to be added to my `buildDecisionTree()` function).

Note that for the Congress data set this likely won't boost your performancy by that much (you're unlikely to get more than another 1% on average), but for some problems it makes a more significant difference.
***

In [ ]:
# OPTIONALLY: put extra code below if you want to do the extension